# 2차 전처리 — 토픽 클러스터링 (영문)

**목적**: 1차 전처리 결과(`ENG_1st_contents.csv`)를 입력으로 받아 토픽별로 클러스터링하고, 클러스터의 내용을 직접 확인해 **유효한 토픽만 남긴** `ENG_2nd_contents.csv`를 생성합니다.

## 임베딩 모델 선택

영어 리뷰/댓글 도메인의 토픽 클러스터링에는 **문장 단위 의미 임베딩**이 핵심입니다.

| 모델 | 장점 | 단점 |
| --- | --- | --- |
| `sentence-transformers/all-MiniLM-L6-v2` | 가볍고 빠름 (384d) | 품질은 mpnet 대비 약간 낮음 |
| **`sentence-transformers/all-mpnet-base-v2`** | **768d, MTEB 영어 의미유사도 SBERT 계열 최상위권 — 채택** | 속도는 MiniLM보다 느림 |
| pure RoBERTa | 일반 문장 단위 표현 가능 | 폀링을 직접 해야 하고 SBERT 대비 클러스터링 품질이 낮음 |

`BERTopic`은 *임베딩 → UMAP → HDBSCAN → c-TF-IDF 키워드 추출*을 한 번에 처리해 주므로 "클러스터 + 대표 키워드 + 시각화"를 한 노트북에서 보며 유효 클러스터를 고르기 적합합니다.

## 파이프라인
1. `ENG_1st_contents.csv` 로드
2. `all-mpnet-base-v2`로 임베딩 계산 (`.npy` 캐시)
3. UMAP + HDBSCAN + CountVectorizer 구성
4. BERTopic 학습
5. 토픽 개요 출력 + 토픽별 샘플 문장 검토
6. 유효 토픽 ID 지정
7. 필터링 후 `ENG_2nd_contents.csv` 저장

## 0. 의존성 설치 (필요 시 주석 해제)

In [3]:
!pip install -q bertopic sentence-transformers umap-learn hdbscan scikit-learn pandas numpy tqdm plotly

## 1. Imports

In [4]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "text_preprocessing":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "text_preprocessing").exists():
    PROJECT_ROOT = PROJECT_ROOT

OUT_DIR = PROJECT_ROOT / "out" / "text_preprocessing"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. 설정

In [6]:
INPUT_CSV = OUT_DIR / "ENG_1st_contents.csv"
OUTPUT_CSV = OUT_DIR / "ENG_2nd_contents.csv"
EMBED_CACHE = OUT_DIR / "ENG_1st_embeddings.npy"

EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"
RANDOM_SEED = 42

if torch.cuda.is_available():
    device_str = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device_str = "mps"
else:
    device_str = "cpu"

print(f"project root: {PROJECT_ROOT}")
print(f"device: {device_str}")

device: cpu


## Step 1 — `ENG_1st_contents.csv` 로드

컬럼명이 `content`(현재 파일) / `contents`(새 포맷) 둘 다 올 수 있으므로 방어적으로 읽습니다.

In [ ]:
df = pd.read_csv(INPUT_CSV)
print("shape:", df.shape)
print("columns:", list(df.columns))

if "contents" not in df.columns and "content" in df.columns:
    df = df.rename(columns={"content": "contents"})

df = df[["contents"]].dropna()
df["contents"] = df["contents"].astype(str).str.strip()
df = df[df["contents"].str.len() > 0].reset_index(drop=True)

print("after cleaning:", df.shape)
df.head()

## Step 2 — 임베딩 모델 로드

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL, device=device_str)
print("embedding dim:", embedder.get_sentence_embedding_dimension())

## Step 3 — 임베딩 계산 (캐시)

27만 행 규모에서는 재실행 시 시간을 아끼기 위해 `.npy`로 캐시합니다. 캠시 파일이 존재하고 행 수가 일치하면 그대로 로드.

In [ ]:
docs = df["contents"].tolist()

if os.path.exists(EMBED_CACHE):
    cached = np.load(EMBED_CACHE)
    if cached.shape[0] == len(docs):
        embeddings = cached
        print(f"loaded cached embeddings: {embeddings.shape}")
    else:
        print(f"cache mismatch ({cached.shape[0]} != {len(docs)}). recomputing...")
        embeddings = None
else:
    embeddings = None

if embeddings is None:
    embeddings = embedder.encode(
        docs,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )
    np.save(EMBED_CACHE, embeddings)
    print(f"saved embeddings → {EMBED_CACHE} ({embeddings.shape})")

## Step 4 — UMAP / HDBSCAN / Vectorizer 구성

- `UMAP`: 고차원 임베딩을 5차원으로 압축 (cosine 거리).
- `HDBSCAN`: 밀도 기반 클러스터링. `min_cluster_size`가 클수록 포괄적 토픽.
- `CountVectorizer`: c-TF-IDF 용 영어 불용어 제거, 1~2-gram, 최소 등장 회수 5.

In [ ]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_SEED,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
)

## Step 5 — BERTopic 학습

이미 계산해둔 `embeddings`를 전달해 재임베딩을 피합니다.

In [ ]:
topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs, embeddings)
df["topic"] = topics
print("unique topics:", len(set(topics)))

## Step 6 — 토픽 개요

각 토픽의 ID, 구성 문서 수, 대표 키워드(`Name`)를 한눈에 확인. 토픽 `-1`은 노이즈 클러스터입니다.

In [ ]:
topic_info = topic_model.get_topic_info()
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)
topic_info

## Step 7-1 — 토픽별 샘플 문장 수동 확인 헬퍼

토픽 ID를 넘기면 해당 토픽의 대표 키워드와 랜덤 샘플 문장을 출력합니다. 아래 셀에서 토픽별로 돌려보며 유효성을 판단하세요.

In [ ]:
def show_topic_examples(topic_id: int, n: int = 10, seed: int = 0) -> pd.DataFrame:
    keywords = topic_model.get_topic(topic_id)
    if keywords:
        kw_str = ", ".join([w for w, _ in keywords[:10]])
        print(f"[topic {topic_id}] keywords: {kw_str}")
    else:
        print(f"[topic {topic_id}] (no keywords)")

    subset = df[df["topic"] == topic_id]
    print(f"size: {len(subset)} docs")
    if len(subset) == 0:
        return subset
    return subset.sample(min(n, len(subset)), random_state=seed)[["contents"]]

In [ ]:
show_topic_examples(0, n=10)

In [ ]:
show_topic_examples(1, n=10)

In [ ]:
show_topic_examples(2, n=10)

## Step 7-2 — 토픽별 샘플 문장 csv 파일 생성 코드

토픽 ID를 넘기면 해당 토픽의 대표 키워드와 랜덤 샘플 문장을 20개씩 저장합니다.

저장된 topic_examples_20_wide.csv 파일을 GPT, Geminai, Claude에게 전송하고, 주제와 관련된 토픽 번호만 골라내 달라고 하면 된다.

AI에게 부탁할 때, 파일을 전부 읽고 처리해달라고 요청하는 것을 권장함.

In [ ]:
import pandas as pd
from pathlib import Path
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter


def export_topic_examples_wide_excel(
    df: pd.DataFrame,
    topic_model,
    n: int = 20,
    seed: int = 0,
    csv_long_path: str = "topic_examples_20.csv",
    csv_wide_path: str = "topic_examples_20_wide.csv",
    excel_path: str = "topic_examples_20_wide.xlsx",
    include_outlier: bool = True
):
    """
    각 topic_id별로 글 n개씩 샘플링한 뒤,
    1) 세로형 CSV
    2) 가로형 CSV
    3) 엑셀에서 보기 좋은 가로형 XLSX
    를 모두 저장합니다.

    세로형 CSV 컬럼:
    - topic_id
    - topic_keywords
    - example_no
    - contents

    가로형 XLSX 컬럼:
    - topic_id
    - topic_keywords
    - example_01 ~ example_20
    """

    rows = []

    topic_ids = sorted(df["topic"].dropna().unique())

    if not include_outlier:
        topic_ids = [t for t in topic_ids if int(t) != -1]

    for topic_id in topic_ids:
        topic_id = int(topic_id)

        keywords = topic_model.get_topic(topic_id)
        if keywords:
            kw_str = ", ".join([w for w, _ in keywords[:10]])
        else:
            kw_str = ""

        subset = df[df["topic"] == topic_id]

        print(f"[topic {topic_id}] keywords: {kw_str}")
        print(f"size: {len(subset)} docs")

        if len(subset) == 0:
            continue

        sampled_df = subset.sample(
            min(n, len(subset)),
            random_state=seed
        )[["contents"]].reset_index(drop=True)

        for i, row in sampled_df.iterrows():
            rows.append({
                "topic_id": topic_id,
                "topic_keywords": kw_str,
                "example_no": i + 1,
                "contents": row["contents"]
            })

    topic_examples_df = pd.DataFrame(rows)

    # 1. 세로형 CSV 저장
    topic_examples_df.to_csv(
        csv_long_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"세로형 CSV 저장 완료: {csv_long_path}")

    # 2. 가로형 데이터프레임 생성
    wide_df = (
        topic_examples_df
        .pivot(
            index=["topic_id", "topic_keywords"],
            columns="example_no",
            values="contents"
        )
        .reindex(columns=range(1, n + 1))
        .reset_index()
    )

    wide_df.columns = (
        ["topic_id", "topic_keywords"] +
        [f"example_{i:02d}" for i in range(1, n + 1)]
    )

    # 3. 가로형 CSV 저장
    wide_df.to_csv(
        csv_wide_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"가로형 CSV 저장 완료: {csv_wide_path}")

    # 4. 보기 좋은 XLSX 저장
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        wide_df.to_excel(
            writer,
            index=False,
            sheet_name="topic_examples"
        )

        wb = writer.book
        ws = wb["topic_examples"]

        # 첫 행, 앞 두 컬럼 고정
        ws.freeze_panes = "C2"

        # 헤더 스타일
        header_fill = PatternFill(
            fill_type="solid",
            fgColor="D9EAF7"
        )
        header_font = Font(
            bold=True,
            size=11
        )

        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )

        # 컬럼 너비
        ws.column_dimensions["A"].width = 12
        ws.column_dimensions["B"].width = 45

        for col_idx in range(3, 3 + n):
            col_letter = get_column_letter(col_idx)
            ws.column_dimensions[col_letter].width = 50

        # 셀 줄바꿈, 위쪽 정렬
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=True
                )

        # 행 높이
        for row_idx in range(2, ws.max_row + 1):
            ws.row_dimensions[row_idx].height = 130

        # 헤더 행 높이
        ws.row_dimensions[1].height = 25

        # 자동 필터
        ws.auto_filter.ref = ws.dimensions

    print(f"엑셀 저장 완료: {excel_path}")
    print(f"총 저장 글 수: {len(topic_examples_df)}")
    print(f"저장 topic 수: {topic_examples_df['topic_id'].nunique()}")

    return topic_examples_df, wide_df


topic_examples_df, wide_df = export_topic_examples_wide_excel(
    df=df,
    topic_model=topic_model,
    n=20,
    seed=0,
    csv_long_path=str(OUT_DIR / "topic_examples_20.csv"),
    csv_wide_path=str(OUT_DIR / "topic_examples_20_wide.csv"),
    excel_path=str(OUT_DIR / "topic_examples_20_wide.xlsx"),
    include_outlier=True
)

## Step 8 — 시각화 (선택)

전체 토픽 구조/키워드를 인터랙티브하게 살펴볼 수 있습니다. 대형 데이터셋에서는 렌더링이 느릴 수 있으므로 필요 시에만 실행.

In [ ]:
# topic_model.visualize_topics()

In [ ]:
# topic_model.visualize_barchart(top_n_topics=20)

## Step 9 — 유효 토픽 ID 선정

위 검토 결과를 보고 **냄새 도메인에 유효한 토픽**만 골라 아래 리스트에 절어넣으세요.

- 노이즈 토픽 `-1`은 기본적으로 제외됩니다.
- 예: `VALID_TOPIC_IDS = [0, 2, 5, 7, 11]`

In [ ]:
VALID_TOPIC_IDS: list[int] = [
    # TODO: 위 셀들 검토 후 유효한 토픽 ID 입력
]

print(f"selected {len(VALID_TOPIC_IDS)} topics")

## Step 10 — 유효 클러스터 필터링

In [ ]:
before = len(df)
df_valid = df[df["topic"].isin(VALID_TOPIC_IDS)].reset_index(drop=True)
print(f"filter by valid topics: {before} → {len(df_valid)}")
df_valid["topic"].value_counts()

## Step 11 — 결과 저장 → `ENG_2nd_contents.csv`

`topic` 컬럼은 추적/분석을 위해 함께 저장. 순수 `contents`만 원한다면 아래 주석 해제.

In [ ]:
out_df = df_valid[["contents", "topic"]].copy()
# out_df = df_valid[["contents"]].copy()

out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"saved → {OUTPUT_CSV} ({len(out_df)} rows, {out_df['topic'].nunique() if 'topic' in out_df.columns else '-'} topics)")

## 요약

- `all-mpnet-base-v2`로 문장 임베딩 추출
- BERTopic(UMAP + HDBSCAN + c-TF-IDF)으로 토픽 클러스터링
- 토픽별 키워드·샘플을 검토해 유효한 클러스터만 선별
- 최종 결과: `ENG_2nd_contents.csv`